# Dask Pyarrow String Conversion

![pic1](./stringpic1.png)

![pic2](./stringpic2.png)

In [1]:
import pandas as pd

import lsdb

In [2]:
from hats import HealpixPixel

df = pd.DataFrame({
    "a": [1, 2, 3],
    "b": [4, 5, 6],
    "ra": [10.0, 20.0, 30.0],
    "dec": [-10.0, -20.0, -30.0],
})
df

,a,b,ra,dec
0,1,4,10.0,-10.0
1,2,5,20.0,-20.0
2,3,6,30.0,-30.0


In [3]:
cat = lsdb.from_dataframe(df)

In [4]:
cat

,a,b,ra,dec
npartitions=2,,,,
"Order: 1, Pixel: 16",int64[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow]
"Order: 7, Pixel: 142694",...,...,...,...


In [5]:
res = cat.map_partitions(lambda df: df.assign(c=HealpixPixel(0, 0))).compute()

Computing Catalog:   0%|          | 0/11 [00:00<?, ?it/s]

In [6]:
res.dtypes

a       int64[pyarrow]
b       int64[pyarrow]
ra     double[pyarrow]
dec    double[pyarrow]
c               object
dtype: object

In [7]:
display(type(res["c"].iloc[0]))
res["c"].iloc[0]

hats.pixel_math.healpix_pixel.HealpixPixel

Order: 0, Pixel: 0

In [26]:
lsdb.from_dataframe(res)

ArrowInvalid: Could not convert Order: 0, Pixel: 0 with type HealpixPixel: did not recognize Python value type when inferring an Arrow data type

In [28]:
from_df_cat = lsdb.from_dataframe(res, use_pyarrow_types=False)
from_df_cat

,a,b,ra,dec,c
npartitions=2,,,,,
"Order: 1, Pixel: 16",int64[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],string
"Order: 7, Pixel: 142694",...,...,...,...,...


In [32]:
elem = from_df_cat.compute()["c"].iloc[0]
display(elem)
type(elem)

Computing Catalog:   0%|          | 0/9 [00:00<?, ?it/s]

'Order: 0, Pixel: 0'

str

In [20]:
import dask.dataframe as dd
dd.from_pandas(cat.map_partitions(lambda df: df.assign(c=HealpixPixel(0, 0))).compute(), npartitions=1)

Computing Catalog:   0%|          | 0/11 [00:00<?, ?it/s]

,a,b,ra,dec,c
npartitions=1,,,,,
1176808107119886823,int64[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],string
2510306432296314470,...,...,...,...,...


## After changing dask config

In [9]:
lsdb.from_dataframe(res, use_pyarrow_types=False)

,a,b,ra,dec,c
npartitions=2,,,,,
"Order: 1, Pixel: 16",int64[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],object
"Order: 7, Pixel: 142694",...,...,...,...,...


In [10]:
lsdb.from_dataframe(res)

ArrowInvalid: Could not convert Order: 0, Pixel: 0 with type HealpixPixel: did not recognize Python value type when inferring an Arrow data type

In [12]:
lsdb.from_dataframe(df.assign(c=["some", "string", "values"]))

,a,b,ra,dec,c
npartitions=2,,,,,
"Order: 1, Pixel: 16",int64[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],string[pyarrow]
"Order: 7, Pixel: 142694",...,...,...,...,...


In [13]:
lsdb.from_dataframe(df.assign(c=["some", "string", "values"]), use_pyarrow_types=False)

,a,b,ra,dec,c
npartitions=2,,,,,
"Order: 1, Pixel: 16",int64,int64,float64,float64,object
"Order: 7, Pixel: 142694",...,...,...,...,...


In [15]:
import dask.dataframe as dd
dd.from_pandas(res, npartitions=1)

,a,b,ra,dec,c
npartitions=1,,,,,
1176808107119886823,int64[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],object
2510306432296314470,...,...,...,...,...


In [18]:
dd.from_pandas(df.assign(c=["some", "string", "values"]), npartitions=1)

,a,b,ra,dec,c
npartitions=1,,,,,
0,int64,int64,float64,float64,object
2,...,...,...,...,...
